In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
from zenodo_get import download
import os
from lxml import etree
import zipfile
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from joblib import parallel_backend
from util.preprocessing import TweetPreprocessor 
import joblib
from huggingface_hub import upload_file

# Constant

In [ ]:
DATASETS_DIR = os.path.join(os.getcwd(), "datasets")
MODELS_DIR = os.path.join(os.getcwd(), "models")

# Dataset 100 tweets per author

Pan clef dataset from author profiling task

https://zenodo.org/records/3692340

https://doi.org/10.5281/zenodo.3692340

In [ ]:
download(record_or_doi="3692340", output_dir=DATASETS_DIR)

Remove not used parts

In [ ]:
unused_files_path = [
    "pan19-author-profiling-earlybirds-20190320.zip",
    "pan19-author-profiling-earlybirds-20190320.zip",
    "pan19-author-profiling-20200229.zip"
]

In [ ]:
for file in unused_files_path:
    file_path = os.path.join(DATASETS_DIR, file)
    if os.path.exists(file_path):
        if os.path.isfile(file_path):
            os.remove(file_path)
        else:
            os.rmdir(file_path)

In [ ]:
files_for_extract = [
    "pan19-author-profiling-training-dataset-2019-02-18.zip",
    "pan19-author-profiling-test-2019-04-29.zip"
]

In [ ]:
for file in files_for_extract:
    file_path = os.path.join(DATASETS_DIR, file)
    if os.path.exists(file_path) and zipfile.is_zipfile(file_path):
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(DATASETS_DIR)
        os.remove(file_path)

In [ ]:
training_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-training-2019-02-18", "en")
if not os.path.exists(training_path):
    raise FileNotFoundError(f"Training path not found: {training_path}")
test_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-test-2019-04-29", "en")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Test path not found: {test_path}")
test_truth_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-test-2019-04-29", "en.txt")
if not os.path.exists(test_truth_path):
    raise FileNotFoundError(f"Test truth path not found: {test_truth_path}")

In [ ]:
train_df = pd.DataFrame()
for file in os.listdir(training_path):
    if not file.endswith(".xml"):
        continue
    try:
        tmp_df = pd.read_xml(os.path.join(training_path, file))
        tmp_df["id"] = file.split(".")[0]
        train_df = pd.concat([train_df, tmp_df], ignore_index=True)
    except ValueError as e:
        print(f"Error reading file {file}: {e}")

truth_train_df = pd.read_csv(os.path.join(training_path, "truth-train.txt"), sep=":::", header=None, names=["id", "who","gender_label"], engine="python")
train_df = train_df.merge(truth_train_df, on="id", how="left")
train_df.head()

In [ ]:
test_df = pd.DataFrame()
for file in os.listdir(test_path):
    if not file.endswith(".xml"):
        continue
    try:
        tmp_df = pd.read_xml(os.path.join(test_path, file))
        tmp_df["id"] = file.split(".")[0]
        test_df = pd.concat([test_df, tmp_df], ignore_index=True)
    except ValueError as e:
        print(f"Error reading file {file}: {e}")

truth_test_df = pd.read_csv(
    test_truth_path,
    sep=":::",
    header=None,
    names=["id", "who", "gender_label"],
    engine="python",
)
test_df = test_df.merge(truth_test_df, on="id", how="left")
test_df.head()

In [ ]:
text_col = "text" if "text" in train_df.columns else "document"

train_eval_df = train_df[[text_col, "gender_label"]].dropna().copy()
test_eval_df = test_df[[text_col, "gender_label"]].dropna().copy()

x_train = train_eval_df[text_col].astype(str)
y_train = train_eval_df["gender_label"]
x_test = test_eval_df[text_col].astype(str)
y_test = test_eval_df["gender_label"]

# Baseline pipeline from PAN-CLEF

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),

    ("features", FeatureUnion([
        ("tfidf_word",
         TfidfVectorizer(
             analyzer="word",
             ngram_range=(1,3),     # English
             min_df=2,
             max_df=1.0,
             sublinear_tf=True,
             lowercase=True,
             norm="l2"
         )
        ),

        ("tfidf_char",
         TfidfVectorizer(
             analyzer="char",
             ngram_range=(3,5),
             min_df=2,
             max_df=1.0,
             sublinear_tf=True,
             lowercase=True,
             norm="l2"
         )
        ),
    ])), # type: ignore

    ("svd",
     TruncatedSVD(
         n_components=300,
         random_state=int(os.getenv("RANDOM_SEED", 880055535))
     )
    ),

    ("clf",
     LinearSVC(
         C=1.0,
         random_state=int(os.getenv("RANDOM_SEED", 880055535))
     )
    ),
])

# Training model on 100 tweets per author

In [ ]:
pipeline.fit(x_train, y_train)

# Importing 1 tweet per author model

Must have already trained model with one tweet per author

In [ ]:
one_tweet_model_path = os.path.join(MODELS_DIR, "model.joblib")

if os.path.exists(one_tweet_model_path):
    model = joblib.load(one_tweet_model_path)
else:
    raise FileNotFoundError(f"Model file not found: {one_tweet_model_path}")

# Comparison of 1 tweet vs 100 tweets per author models

## Test on 100 tweets per author dataset

In [ ]:
y_pred_100 = pipeline.predict(x_test)
y_pred_1 = model.predict(x_test)

labels = sorted(y_test.dropna().unique())

print("100 tweets per author model")
print(classification_report(y_test, y_pred_100, digits=4))

print("1 tweet per author model")
print(classification_report(y_test, y_pred_1, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_100,
    labels=labels,
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("100 tweets per author")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_1,
    labels=labels,
    cmap="Blues",
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("1 tweet per author")

plt.tight_layout()
plt.show()

## Test on 1 tweet per author dataset

In [ ]:
test_df_1 = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_test.csv"))

In [ ]:
x_test_1 = test_df_1[text_col]
y_test_1 = test_df_1["gender_label"]

In [ ]:
y_pred_100 = pipeline.predict(x_test_1)
y_pred_1 = model.predict(x_test_1)

labels = sorted(y_test_1.dropna().unique())

print("100 tweets per author model")
print(classification_report(y_test_1, y_pred_100, digits=4))

print("1 tweet per author model")
print(classification_report(y_test_1, y_pred_1, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test_1,
    y_pred_100,
    labels=labels,
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("100 tweets per author")

ConfusionMatrixDisplay.from_predictions(
    y_test_1,
    y_pred_1,
    labels=labels,
    cmap="Blues",
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("1 tweet per author")

plt.tight_layout()
plt.show()